# **Does Housing Supply Constrain Home Prices? Evidence from U.S. Metro Areas**

### ECON 320 Lab — Final Project
**Student names:** Tessa Butler, Siya Kumar, Ania Ting, Naman Keswani, Shunji Lewandowski  
**Lab Section:** [#]  

---
## Abstract
We study whether new residential construction is associated with lower home prices across U.S. Metropolitan Statistical Areas (MSAs) using a **pooled cross-section** of ~920 MSAs over 2019–2024 (excluding 2020). The unit of observation is an MSA-year pair ($N \approx 4{,}600$). The dependent variable is the log of the FHFA All-Transactions House Price Index. The main regressor is log building permits (Census BPS). The baseline specification is:

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \beta_2 \ln(\text{Income}_{it}) + \beta_3 \ln(\text{Pop}_{it}) + \gamma\,\text{Metro}_i + \sum_t \delta_t\,\text{Year}_t + u_{it}$$

We find that **[main result — fill after running]**. Comparing the bivariate ("short") and full ("long") regressions, $\hat{\beta}_1$ moves **[direction]**, consistent with **[upward/downward] OVB** from demand-side factors. Results are **[robust/not robust]** to alternative DVs and sample trimming. We test for heteroskedasticity (Breusch–Pagan and White tests) and multicollinearity (VIF). Our findings suggest **[policy implication]**.

---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats import diagnostic as sm_diagnostic
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---
## 1. Introduction

- **Research question:** Is greater housing supply — measured by new residential building permits — associated with lower home prices across U.S. metropolitan areas?

- **Why it matters:** The U.S. housing supply gap widened to an estimated 4.03 million homes in 2025 (Realtor.com, 2026). Over 42 million Americans spend more than 30% of income on housing. Standard supply-and-demand theory predicts that constrained supply pushes prices above competitive equilibrium: when fewer homes are built relative to demand, prices rise. This is the core mechanism we test.

- **Preview of approach:** We use FHFA house price index data merged with Census building permits and ACS demographic controls (income, population) across ~920 MSAs and 5 years (2019, 2021–2024). We estimate a **log-log OLS** model where the coefficient on log permits is the **price-supply elasticity**. We progressively add controls to demonstrate OVB (Week 7), test for multicollinearity via VIF (Week 6), check heteroskedasticity using BP/White tests (Week 10), test joint significance of year dummies via the $F$-test (Week 9), and use HC0 robust standard errors throughout. We discuss reverse causality and the potential role of instrumental variables (Week 11) as a limitation.

---
## 2. Data and Summary Statistics

**Data sources & unit of observation:**

Our unit of observation is an **MSA-year pair**. We pool ~920 MSAs across 5 years (2019, 2021–2024; 2020 is excluded because the Census Bureau did not release ACS 1-year estimates that year due to COVID data-collection disruptions). This gives us a **pooled cross-section** (Wooldridge Ch. 1 §3.iii).

| Source | Variable | Description |
|--------|----------|-------------|
| FHFA | `HPI` | All-Transactions House Price Index (annual, CBSA-level). Repeat-sales index using Fannie Mae/Freddie Mac + FHA + county recorder data. |
| FHFA | `Ann_Chg_Pct` | Year-over-year percent change in HPI. Used in robustness. |
| Census BPS | `total_permits` | New privately-owned residential units authorized by building permit (CBSA, Jan 2026 monthly snapshot). |
| Census BPS | `metro_dummy` | = 1 if Metropolitan, = 0 if Micropolitan. Binary qualitative variable (Week 5). |
| ACS 1-Year | `median_household_income` | Median household income (Table B19013). Time-varying by MSA-year. |
| ACS 1-Year | `total_population` | Total population (Table B01003). Time-varying by MSA-year. |
| ACS 1-Year | `housing_units` | Total housing units (Table B25001). Time-varying by MSA-year. |

**Variable definitions:**

| Role | Variable | Construction | Expected sign |
|------|----------|--------------|---------------|
| DV ($y$) | `log_hpi` | $\ln(\text{HPI}_{it})$ | — |
| Main regressor ($x$) | `log_permits` | $\ln(\text{Permits}_i + 1)$ | Negative ($-$): more supply → lower prices |
| Control | `log_income` | $\ln(\text{Median HH Income}_{it})$ | Positive ($+$): higher income → higher demand → higher prices |
| Control | `log_pop` | $\ln(\text{Population}_{it})$ | Ambiguous: larger cities may have both more demand and more supply |
| Control | `metro_dummy` | 1 = Metro, 0 = Micro | Positive ($+$): metros have structurally higher prices |
| Controls | `Year_2021` … `Year_2024` | Year dummies, base = 2019 | Positive: national price appreciation over time |

**Why log-log?** (Wooldridge, Appendix A.4)
1. Reduces right-skew in HPI and permits.
2. $\beta_1$ is directly interpretable as an **elasticity**: a 1% increase in permits is associated with a $\beta_1$% change in HPI.

**Why `+1` before log?** Some MSAs have zero permits in the Jan 2026 snapshot. Since $\ln(0)$ is undefined, we add 1. We verify in Robustness Check 2 that results hold when dropping zero-permit MSAs instead.

### 2.1 Load and prepare data

In [ ]:
hpi_raw = pd.read_excel('data/hpi_at_cbsa.xlsx', skiprows=5, header=None)
hpi_raw.columns = ['CBSA', 'Name_hpi', 'Year', 'Ann_Chg_Pct', 'HPI', 'HPI_1990', 'HPI_2000']

for col in ['CBSA', 'Year', 'Ann_Chg_Pct', 'HPI']:
    hpi_raw[col] = pd.to_numeric(hpi_raw[col], errors='coerce')

hpi = hpi_raw[
    (hpi_raw['CBSA'] >= 10000) &
    (hpi_raw['Year'].isin([2019, 2021, 2022, 2023, 2024]))
][['CBSA', 'Year', 'HPI', 'Ann_Chg_Pct']].copy()

hpi['CBSA'] = hpi['CBSA'].astype(int)
hpi = hpi.dropna(subset=['HPI']).reset_index(drop=True)
print(f'FHFA HPI: {len(hpi)} MSA-year obs | {hpi["CBSA"].nunique()} MSAs | Years: {sorted(hpi["Year"].unique())}')
hpi.head()

In [ ]:
acs = pd.read_csv('data/merged_2010_2024.csv')
print(f'ACS raw: {acs.shape}')
print(acs.columns.tolist())
acs.head()

In [ ]:
acs['CBSA'] = acs['GEO_ID'].astype(str).str[-5:].astype(int)
acs['year'] = pd.to_numeric(acs['year'], errors='coerce')

for col in ['housing_units', 'median_household_income', 'total_population']:
    acs[col] = pd.to_numeric(acs[col], errors='coerce')

acs = acs[
    (acs['CBSA'] >= 10000) &
    (acs['year'].isin([2019, 2021, 2022, 2023, 2024])) &
    (acs['median_household_income'] > 0) &
    (acs['total_population'] > 0)
].reset_index(drop=True)

print(f'ACS filtered: {len(acs)} MSA-year obs | {acs["CBSA"].nunique()} MSAs')
acs[['CBSA', 'NAME', 'year', 'housing_units', 'median_household_income', 'total_population']].head()

In [ ]:
perm_raw = pd.read_excel('data/cbsamonthly_202601.xls', engine='xlrd', skiprows=6, header=None)
data_rows = perm_raw.iloc[2:].reset_index(drop=True)

perm = pd.DataFrame({
    'CBSA': pd.to_numeric(data_rows[1], errors='coerce'),
    'metro_code': pd.to_numeric(data_rows[3], errors='coerce'),
    'total_permits': pd.to_numeric(data_rows[4], errors='coerce')
})
perm = perm.dropna(subset=['CBSA', 'total_permits'])
perm['CBSA'] = perm['CBSA'].astype(int)
perm['metro_dummy'] = (perm['metro_code'].isin([2, 4])).astype(int)

print(f'Permits: {len(perm)} CBSAs | Metro: {perm["metro_dummy"].sum()}, Micro: {(1-perm["metro_dummy"]).sum()}')
perm.head()

> **`metro_dummy` (Week 5 — dummy variables):** Census BPS uses `metro_code` 2/4 = Metropolitan, 5 = Micropolitan. We create a binary indicator where 1 = Metro, 0 = Micro. This captures the structural price-level difference between larger and smaller urban areas. We drop the base category (Micro) to avoid the **dummy variable trap** (perfect multicollinearity — Week 5/6).

In [ ]:
df = hpi.merge(
    acs[['CBSA', 'year', 'housing_units', 'median_household_income', 'total_population']],
    left_on=['CBSA', 'Year'], right_on=['CBSA', 'year'], how='inner'
).merge(
    perm[['CBSA', 'total_permits', 'metro_dummy']],
    on='CBSA', how='inner'
)

df = df.drop(columns=['year']).reset_index(drop=True)
print(f'Merged: {len(df)} MSA-year obs | {df["CBSA"].nunique()} unique MSAs | Years: {sorted(df["Year"].unique())}')

In [ ]:
df['log_hpi']     = np.log(df['HPI'])
df['log_permits'] = np.log(df['total_permits'] + 1)
df['log_income']  = np.log(df['median_household_income'])
df['log_pop']     = np.log(df['total_population'])

year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
df = pd.concat([df, year_dummies], axis=1)

df = df.dropna(subset=['log_hpi', 'log_permits', 'log_income', 'log_pop']).reset_index(drop=True)

print(f'Final sample: {len(df)} obs | {df["CBSA"].nunique()} MSAs')
print(f'Zero-permit MSAs: {(df["total_permits"] == 0).sum()} ({(df["total_permits"]==0).mean()*100:.1f}%)')
print(f'Year dummies: {[c for c in year_dummies.columns]} (base = 2019)')
df[['CBSA','Year','HPI','log_hpi','total_permits','log_permits','median_household_income','log_income','total_population','log_pop','metro_dummy']].head(10)

> **Year dummies (Week 5):** We use `pd.get_dummies(drop_first=True)` to create indicator variables for 2021–2024, omitting 2019 as the **base category**. This avoids the **dummy variable trap** — if we included all 5 year dummies plus an intercept, they would sum to 1 for every observation, creating **perfect multicollinearity** (the design matrix $X$ would not have full rank, and $(X'X)^{-1}$ would not exist — Week 6). Each year coefficient $\delta_t$ measures how much higher/lower log HPI was in year $t$ relative to 2019, holding all other variables constant.

### 2.2 Descriptive statistics *(table + figure)*

In [ ]:
desc_vars = ['HPI', 'Ann_Chg_Pct', 'total_permits', 'median_household_income',
             'total_population', 'log_hpi', 'log_permits', 'log_income', 'log_pop', 'metro_dummy']
desc = df[desc_vars].describe().T[['count','mean','std','min','25%','50%','75%','max']].round(3)
print(desc.to_string())

> **Observation:** [After running, note: mean HPI, range of permits, share of metro observations, and whether the raw variables are right-skewed (compare mean vs median). Right-skew motivates the log transformation.]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

axes[0,0].hist(df['HPI'], bins=50, color='steelblue', edgecolor='white', lw=0.3)
axes[0,0].set_xlabel('HPI'); axes[0,0].set_title('Fig 1a: HPI (levels)')

axes[0,1].hist(df['log_hpi'], bins=50, color='steelblue', edgecolor='white', lw=0.3)
axes[0,1].set_xlabel('Log(HPI)'); axes[0,1].set_title('Fig 1b: Log(HPI)')

axes[0,2].hist(df['log_permits'], bins=50, color='coral', edgecolor='white', lw=0.3)
axes[0,2].set_xlabel('Log(Permits + 1)'); axes[0,2].set_title('Fig 1c: Log(Permits)')

axes[1,0].hist(df['log_income'], bins=50, color='seagreen', edgecolor='white', lw=0.3)
axes[1,0].set_xlabel('Log(Income)'); axes[1,0].set_title('Fig 1d: Log(Income)')

axes[1,1].hist(df['log_pop'], bins=50, color='mediumpurple', edgecolor='white', lw=0.3)
axes[1,1].set_xlabel('Log(Population)'); axes[1,1].set_title('Fig 1e: Log(Pop)')

metro_cts = df['metro_dummy'].value_counts().sort_index()
axes[1,2].bar(['Micro (0)', 'Metro (1)'], metro_cts.values, color=['coral','steelblue'], edgecolor='k', lw=0.3)
axes[1,2].set_ylabel('Count'); axes[1,2].set_title('Fig 1f: Metro/Micro Split')

plt.tight_layout()
plt.savefig('fig1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

slope, intercept, r, pv, _ = stats.linregress(df['log_permits'], df['log_hpi'])
x_line = np.linspace(df['log_permits'].min(), df['log_permits'].max(), 200)
axes[0].scatter(df['log_permits'], df['log_hpi'], alpha=0.1, s=8, color='steelblue', edgecolors='none')
axes[0].plot(x_line, slope * x_line + intercept, color='firebrick', lw=2)
axes[0].set_xlabel('Log(Permits + 1)'); axes[0].set_ylabel('Log(HPI)')
axes[0].set_title('Fig 2a: Bivariate — Supply vs. Prices')
axes[0].text(0.04, 0.93, f'r = {r:.3f}, p = {pv:.2e}', transform=axes[0].transAxes, fontsize=9)

slope2, _, r2, pv2, _ = stats.linregress(df['log_income'], df['log_hpi'])
axes[1].scatter(df['log_income'], df['log_hpi'], alpha=0.1, s=8, color='seagreen', edgecolors='none')
x2 = np.linspace(df['log_income'].min(), df['log_income'].max(), 200)
axes[1].plot(x2, slope2*x2 + (df['log_hpi'].mean()-slope2*df['log_income'].mean()), color='firebrick', lw=2)
axes[1].set_xlabel('Log(Median Income)'); axes[1].set_ylabel('Log(HPI)')
axes[1].set_title('Fig 2b: Income vs. Prices (demand side)')
axes[1].text(0.04, 0.93, f'r = {r2:.3f}, p = {pv2:.2e}', transform=axes[1].transAxes, fontsize=9)

plt.tight_layout()
plt.savefig('fig2_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

> **Key observation for OVB:** If the raw correlation between log_permits and log_hpi is **positive**, this is **counterintuitive** under supply-demand theory and is a textbook signal of **omitted variable bias** (Week 7). High-demand metros (high income, large population) attract both more construction AND sustain higher prices. The bivariate correlation confounds the supply effect with the demand effect. This is exactly what Models 2 and 3 are designed to address.

---
## 3. Empirical Strategy

We estimate three nested OLS specifications. Comparing them reveals the direction and magnitude of omitted variable bias (Week 7).

**Model 1 — Bivariate ("short" regression, OVB benchmark):**

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + u_{it} \tag{1}$$

**Model 2 — Year dummies + metro dummy (pooled cross-section controls):**

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \gamma\,\text{Metro}_i + \sum_{t=2021}^{2024} \delta_t\,\mathbf{1}\{\text{Year}=t\} + u_{it} \tag{2}$$

**Model 3 — Full specification (demand-side controls):**

$$\ln(\text{HPI}_{it}) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \beta_2 \ln(\text{Income}_{it}) + \beta_3 \ln(\text{Pop}_{it}) + \gamma\,\text{Metro}_i + \sum_{t} \delta_t\,\text{Year}_t + u_{it} \tag{3}$$

---

### Interpretation of $\beta_1$ (Wooldridge Appendix A.4)

In a log-log model, $\beta_1$ is the **elasticity** of HPI with respect to permits:

> *"A 1% increase in building permits is associated with a $\beta_1$% change in the house price index, holding all other variables constant."*

Supply-demand theory predicts **$\beta_1 < 0$**: more supply → lower prices, ceteris paribus.

---

### Identification concerns

**Omitted Variable Bias (OVB) — Week 7:**

The bias in the short regression (Model 1) follows the OVB formula:

$$\text{Bias}(\hat{\beta}_1^{\text{short}}) \approx \beta_2 \cdot \tilde{\delta}_1$$

where $\beta_2$ is the true effect of Income on HPI (positive: higher income → higher prices) and $\tilde{\delta}_1$ is the slope from regressing Income on Permits (positive: richer metros build more). Since both are positive, the bias is **positive**, pushing $\hat{\beta}_1$ upward (less negative or even positive).

*Prediction:* $\hat{\beta}_1$ should become **more negative** when we move from Model 1 → Model 3.

**Multicollinearity — Week 6:**

`log_permits` and `log_pop` will be positively correlated — large cities permit more units. This does **not** bias OLS (Gauss-Markov — OLS remains BLUE), but inflates standard errors. We diagnose with VIF (rule of thumb: VIF $> 10$ is concerning).

**Perfect multicollinearity — Week 5/6:**

Using `drop_first=True` for year dummies avoids the dummy variable trap. If all year dummies were included with an intercept, the columns of $X$ would be linearly dependent and $(X'X)^{-1}$ would not exist.

**Heteroskedasticity — Week 10:**

Under heteroskedasticity ($\text{Var}(u_i | X_i) \neq \sigma^2$), classical SEs are invalid. We use **HC0** (White/Huber/Eicker) robust SEs via `.fit(cov_type="HC0")` and run Breusch–Pagan and White formal tests.

**Reverse causality (Week 11, discussed as limitation):**

Higher prices may attract more construction (developers respond to profits). This means permits are potentially **endogenous**. An instrumental variable (e.g., geographic land constraints, zoning strictness) would address this, but we do not implement IV here — we note it as a limitation.

**Estimation:** OLS with HC0 robust standard errors throughout.

### 3.1 Estimation details

In [ ]:
y = df['log_hpi']
year_cols = [c for c in df.columns if c.startswith('Year_')]

X1 = sm.add_constant(df[['log_permits']])
model1 = sm.OLS(y, X1).fit(cov_type='HC0')
print('=== MODEL 1: Bivariate (Short Regression) ===')
print(model1.summary())

In [ ]:
X2_vars = ['log_permits', 'metro_dummy'] + year_cols
X2 = sm.add_constant(df[X2_vars])
model2 = sm.OLS(y, X2).fit(cov_type='HC0')
print('=== MODEL 2: + Year Dummies + Metro Dummy ===')
print(model2.summary())

In [ ]:
X3_vars = ['log_permits', 'log_income', 'log_pop', 'metro_dummy'] + year_cols
X3 = sm.add_constant(df[X3_vars])
model3 = sm.OLS(y, X3).fit(cov_type='HC0')
print('=== MODEL 3: Full Specification ===')
print(model3.summary())

In [ ]:
comp = pd.DataFrame({
    'Model 1\n(Short)': [model1.params.get('log_permits'),
                         model1.bse.get('log_permits'),
                         model1.pvalues.get('log_permits'),
                         model1.rsquared, int(model1.nobs)],
    'Model 2\n(+ Year/Metro)': [model2.params.get('log_permits'),
                                model2.bse.get('log_permits'),
                                model2.pvalues.get('log_permits'),
                                model2.rsquared, int(model2.nobs)],
    'Model 3\n(Full)': [model3.params.get('log_permits'),
                        model3.bse.get('log_permits'),
                        model3.pvalues.get('log_permits'),
                        model3.rsquared, int(model3.nobs)]
}, index=['β₁ (log_permits)', 'SE(β₁) HC0', 'p-value', 'R²', 'N'])

print('=== OVB COMPARISON: Short → Long Regression ===')
print(comp.round(4).to_string())
print(f'\nEmpirical OVB (M1 → M3) = {model1.params["log_permits"] - model3.params["log_permits"]:.4f}')
print(f'Direction: β₁ moved {"downward (more negative)" if model3.params["log_permits"] < model1.params["log_permits"] else "upward"} — '
      f'{"consistent" if model3.params["log_permits"] < model1.params["log_permits"] else "inconsistent"} with upward OVB prediction')

> **OVB analysis (Week 7):** The table above is the core econometric result. Comparing $\hat{\beta}_1$ across the three nested models:
> - Model 1 (short) captures the **raw** correlation between supply and prices — contaminated by demand-side omitted variables.
> - Model 3 (long) controls for income and population, partially absorbing the demand channel.
> - The **empirical OVB** = $\hat{\beta}_1^{\text{short}} - \hat{\beta}_1^{\text{long}}$ measures how much the omitted variables were biasing the short-regression coefficient.

> If this OVB is positive, it confirms our theoretical prediction: demand factors (positively correlated with permits and positively affecting prices) were pushing $\hat{\beta}_1$ upward in the short regression.

### 3.2 Diagnostic awareness

#### Multicollinearity (VIF) — Week 6

In [ ]:
X_vif = df[X3_vars].copy()
X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif_const.values, i+1)
            for i in range(X_vif.shape[1])]
}).round(2)

print('=== Variance Inflation Factors (Week 6) ===')
print(vif_data.to_string(index=False))
print(f'\nRule of thumb: VIF > 10 = problematic multicollinearity')
print(f'Max VIF = {vif_data["VIF"].max():.2f} → {"CONCERN" if vif_data["VIF"].max() > 10 else "OK"}')

> **Interpretation (Week 6):** VIF measures how much the variance of $\hat{\beta}_j$ is inflated due to correlation with other regressors. $\text{VIF}_j = \frac{1}{1 - R_j^2}$ where $R_j^2$ is from regressing $x_j$ on all other regressors. Multicollinearity inflates SEs but does **not** bias OLS — the estimates remain unbiased (Gauss-Markov, Week 3). If VIF is very high, we may consider dropping redundant variables or combining them.

#### Heteroskedasticity — Breusch-Pagan & White Tests (Week 10)

In [ ]:
ols_classic = sm.OLS(y, X3).fit()

bp_lm, bp_p, _, _ = sm_diagnostic.het_breuschpagan(ols_classic.resid, X3)
white_lm, white_p, _, _ = sm_diagnostic.het_white(ols_classic.resid, X3)

alpha = 0.05
print('=== Heteroskedasticity Tests (Week 10) ===')
print(f'Breusch-Pagan: LM = {bp_lm:.3f},  p = {bp_p:.4f}')
print(f'White:         LM = {white_lm:.3f},  p = {white_p:.4f}')
print(f'\nDecision @ {alpha*100:.0f}%: ' + (
    'Reject H₀ (homoskedasticity) → variance NOT constant → HC0 robust SEs justified ✓'
    if (bp_p < alpha or white_p < alpha)
    else 'Cannot reject homoskedasticity — HC0 is precautionary but harmless'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(model3.fittedvalues, model3.resid, alpha=0.1, s=8, color='steelblue', edgecolors='none')
axes[0].axhline(0, color='firebrick', lw=1.5, ls='--')
axes[0].set_xlabel('Fitted Values — Log(HPI)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Fig 3a: Residuals vs. Fitted — Model 3')

axes[1].scatter(df['log_permits'], model3.resid, alpha=0.1, s=8, color='coral', edgecolors='none')
axes[1].axhline(0, color='firebrick', lw=1.5, ls='--')
axes[1].set_xlabel('Log(Permits + 1)')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Fig 3b: Residuals vs. Main Regressor')

plt.tight_layout()
plt.savefig('fig3_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

> **Reading the residual plots (Week 10):** A "funnel" or "megaphone" shape — where residual spread changes with the fitted value or regressor — is visual evidence of heteroskedasticity. [Describe what you see.] Combined with the formal test results above, this [confirms/does not confirm] the need for robust SEs.

#### F-test: Joint Significance of Year Dummies (Week 9)

In [ ]:
X_r = sm.add_constant(df[['log_permits', 'log_income', 'log_pop', 'metro_dummy']])
model_r = sm.OLS(y, X_r).fit()

SSR_R  = model_r.ssr
SSR_UR = model3.ssr
q = len(year_cols)
n_obs, k_ur = int(model3.nobs), X3.shape[1]

F_stat = ((SSR_R - SSR_UR) / q) / (SSR_UR / (n_obs - k_ur))
p_F = 1 - stats.f.cdf(F_stat, q, n_obs - k_ur)

print('=== F-TEST: Joint Significance of Year Dummies (Week 9) ===')
print(f'H₀: δ_2021 = δ_2022 = δ_2023 = δ_2024 = 0  (q = {q} restrictions)')
print(f'H₁: at least one δ_t ≠ 0')
print(f'\nSSR(Restricted)   = {SSR_R:.3f}')
print(f'SSR(Unrestricted) = {SSR_UR:.3f}')
print(f'F = ((SSR_R - SSR_UR)/q) / (SSR_UR/(n-k-1)) = {F_stat:.3f}')
print(f'p-value = {p_F:.2e}')
print(f'\nDecision @ 5%: {"Reject H₀ — year dummies jointly significant" if p_F < 0.05 else "Cannot reject H₀"}')

> **Interpretation (Week 9):** The $F$-test asks whether a *group* of variables (the year dummies) jointly contributes to explaining $y$. We compute $F = \frac{(SSR_R - SSR_{UR})/q}{SSR_{UR}/(n-k-1)}$ manually (as in Lab 9). This is an **exclusion restriction test** — $H_0$ says all year effects are zero. [If rejected: aggregate time trends matter; keeping year dummies is justified.]

> **Note:** This is different from a $t$-test, which tests **one** coefficient. Even if some individual year dummies are insignificant by their $t$-tests, the group can be jointly significant by the $F$-test (Week 9).

---
## 4. Results

**Main result table and interpretation.**

In [ ]:
results = pd.DataFrame(index=['const','log_permits','log_income','log_pop','metro_dummy']+year_cols+['','R²','N'])
for label, mod in [('Model 1', model1), ('Model 2', model2), ('Model 3', model3)]:
    coefs = mod.params.round(4).to_dict()
    ses   = mod.bse.round(4).to_dict()
    col_coef, col_se = [], []
    for v in results.index:
        if v == '': col_coef.append(''); col_se.append('')
        elif v == 'R²': col_coef.append(f'{mod.rsquared:.4f}'); col_se.append('')
        elif v == 'N': col_coef.append(f'{int(mod.nobs)}'); col_se.append('')
        else: col_coef.append(f'{coefs.get(v, "")}'); col_se.append(f'({ses.get(v, "")})' if v in ses else '')
    results[f'{label} β'] = col_coef
    results[f'{label} SE'] = col_se

print('=== REGRESSION RESULTS TABLE ===')
print(results.to_string())

> **Reading the table:**
> - **$\hat{\beta}_1$ (log_permits):** In Model 3, a 1% increase in building permits is associated with a [β₁]% change in HPI, holding income, population, metro status, and year constant.
> - **$\hat{\beta}_2$ (log_income):** A 1% increase in median income → [β₂]% change in HPI. [Expected positive: richer areas have higher demand.]
> - **$\hat{\beta}_3$ (log_pop):** A 1% increase in population → [β₃]% change in HPI.
> - **metro_dummy:** Metros have ~$100 \times \hat{\gamma}$% higher HPI than micros (log-level dummy interpretation — Week 5).
> - **Year dummies:** [Describe trajectory — which year had the biggest price jump?]
> - **$R^2$:** Model 3 explains [value]% of variation — [compare to Model 1].
> - **Statistical significance:** $\hat{\beta}_1$ has $t$ = [value], $p$ = [value] → [significant/not] at 5%.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sm.qqplot(model3.resid, line='s', ax=ax, alpha=0.15, markersize=3)
ax.set_title('Fig 4: Q-Q Plot of Residuals — Model 3')
plt.tight_layout()
plt.savefig('fig4_qq.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Robustness and Limitations

We perform three robustness checks and discuss key limitations.

**Robustness 1:** Alternative DV — annual HPI percent change instead of log level.

In [ ]:
y_rob1 = df['Ann_Chg_Pct'].dropna()
X_rob1 = sm.add_constant(df.loc[y_rob1.index, X3_vars])
rob1 = sm.OLS(y_rob1, X_rob1).fit(cov_type='HC0')
print('=== Robustness 1: DV = Annual HPI % Change ===')
print(f'β₁ = {rob1.params["log_permits"]:.4f}  SE = {rob1.bse["log_permits"]:.4f}  '
      f'p = {rob1.pvalues["log_permits"]:.4f}  R² = {rob1.rsquared:.4f}')

**Robustness 2:** Drop zero-permit MSAs (use strict log instead of log(x+1)).

In [ ]:
df_nz = df[df['total_permits'] > 0].copy()
df_nz['log_permits_strict'] = np.log(df_nz['total_permits'])
X_nz_vars = ['log_permits_strict', 'log_income', 'log_pop', 'metro_dummy'] + year_cols
rob2 = sm.OLS(df_nz['log_hpi'], sm.add_constant(df_nz[X_nz_vars])).fit(cov_type='HC0')
print(f'=== Robustness 2: Drop zeros (N={len(df_nz)}) ===')
print(f'β₁ = {rob2.params["log_permits_strict"]:.4f}  SE = {rob2.bse["log_permits_strict"]:.4f}  '
      f'p = {rob2.pvalues["log_permits_strict"]:.4f}')

**Robustness 3:** Trim top/bottom 2.5% of permits (outlier sensitivity).

In [ ]:
lo, hi = df['log_permits'].quantile(0.025), df['log_permits'].quantile(0.975)
df_trim = df[(df['log_permits'] >= lo) & (df['log_permits'] <= hi)]
rob3 = sm.OLS(df_trim['log_hpi'], sm.add_constant(df_trim[X3_vars])).fit(cov_type='HC0')
print(f'=== Robustness 3: Trimmed 5% (N={len(df_trim)}) ===')
print(f'β₁ = {rob3.params["log_permits"]:.4f}  SE = {rob3.bse["log_permits"]:.4f}  '
      f'p = {rob3.pvalues["log_permits"]:.4f}')

In [ ]:
rob_df = pd.DataFrame({
    'Specification': ['Model 3 (Main)', 'Rob 1: Ann Chg DV', 'Rob 2: Drop zeros', 'Rob 3: Trim 5%'],
    'β₁': [model3.params['log_permits'], rob1.params['log_permits'],
           rob2.params['log_permits_strict'], rob3.params['log_permits']],
    'SE': [model3.bse['log_permits'], rob1.bse['log_permits'],
           rob2.bse['log_permits_strict'], rob3.bse['log_permits']],
    'p': [model3.pvalues['log_permits'], rob1.pvalues['log_permits'],
          rob2.pvalues['log_permits_strict'], rob3.pvalues['log_permits']],
    'R²': [model3.rsquared, rob1.rsquared, rob2.rsquared, rob3.rsquared],
    'N': [int(model3.nobs), int(rob1.nobs), int(rob2.nobs), int(rob3.nobs)]
}).round(4)
print('=== ROBUSTNESS SUMMARY ===')
print(rob_df.to_string(index=False))

> **Robustness assessment:** $\hat{\beta}_1$ [is/is not] stable across specifications. The sign [does/does not] change, and statistical significance [is/is not] maintained. This [strengthens/weakens] confidence in the main result.

**Limitations:**

1. **Reverse causality (Week 11):** Higher prices may attract more construction (developers respond to profit opportunities), making permits **endogenous**. Our OLS captures association, not causation. An instrumental variable — e.g., geographic building constraints (share of undevelopable land) or zoning indices — could address this. We acknowledge this but do not implement IV.

2. **Permits timing:** We use a single monthly snapshot (Jan 2026) for permits, applied across all years. Ideally, annual permit totals matched to each year would allow permits to vary over time within the pooled cross-section.

3. **Not a true panel:** Although the same MSAs appear each year, our key regressor (permits) does not vary over time, so we cannot use fixed-effects estimation to remove time-invariant MSA unobservables.

4. **Remaining OVB:** Zoning restrictiveness, construction costs, land availability, and migration patterns are omitted. These likely correlate with both permits and prices.

---
## 6. Conclusion

Using a pooled cross-section of ~920 U.S. MSAs over 2019–2024 ($N \approx 4{,}600$), we find that building permit activity is **[positively/negatively]** associated with home prices. In our preferred specification (Model 3), a 1% increase in permitted units is associated with a **$\hat{\beta}_1$ = [value]%** change in the FHFA HPI, controlling for income, population, metro status, and year effects.

The shift in $\hat{\beta}_1$ from Model 1 ($[\text{short}]$) to Model 3 ($[\text{long}]$) is consistent with **upward OVB** from demand factors, confirming our theoretical prediction from Week 7. The Breusch–Pagan and White tests [reject/do not reject] homoskedasticity, justifying our use of HC0 robust SEs. VIF values [are/are not] above 10, indicating multicollinearity is [not] a major concern.

**Policy implication:** [If $\beta_1 < 0$: expanding permitting in supply-constrained metros could moderate price appreciation, though the magnitude suggests [limited/substantial] relief. If $\beta_1 \geq 0$: demand-side pressures dominate — policy must address affordability through income support, mortgage access, or direct subsidies alongside supply expansion.]

---
## References
- **FHFA House Price Index:** Federal Housing Finance Agency. *All-Transactions CBSA Annual Data.* https://www.fhfa.gov/data/hpi/datasets
- **Census Building Permits Survey:** U.S. Census Bureau. *CBSA Monthly.* https://www.census.gov/construction/bps
- **American Community Survey:** U.S. Census Bureau. *ACS 1-Year Estimates,* Tables B19013, B25001, B01003. https://data.census.gov
- **Realtor.com.** (2026). *2026 Housing Supply Gap Report.*
- Glaeser, E. L., Gyourko, J., & Saks, R. E. (2005). Why have housing prices gone up? *American Economic Review, 95*(2), 329–333.
- Wooldridge, J. M. (2019). *Introductory Econometrics: A Modern Approach* (7th ed.). Cengage.

---